# บทที่ 6: การพัฒนา Neural Network

ใน Notebook นี้ เราจะเรียนรู้กระบวนการพัฒนา Neural Network ตั้งแต่การเตรียมข้อมูล การเลือกสถาปัตยกรรม การฝึกฝน และ Hyperparameter Tuning

## 1. นำเข้าไลบรารี

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

# ติดตั้งฟอนต์ภาษาไทยสำหรับ Google Colab
import subprocess, glob
subprocess.run(['apt-get', 'install', '-y', '-qq', 'fonts-tlwg-garuda'], 
               capture_output=True)

# ลงทะเบียนฟอนต์โดยตรง
from matplotlib.font_manager import fontManager
for font_file in glob.glob('/usr/share/fonts/truetype/tlwg/*.ttf'):
    fontManager.addfont(font_file)

# ตั้งค่า Seaborn theme และฟอนต์ภาษาไทย
sns.set_theme(style='whitegrid', font='Garuda')
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
%config InlineBackend.figure_format = 'retina'

from sklearn.datasets import load_iris, load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score

np.random.seed(42)

## 2. การเตรียมข้อมูล (Data Preparation)

In [ ]:
# โหลดข้อมูล
data = load_iris()
X, y = data.data, data.target

# แบ่งข้อมูล
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Dataset: Iris")
print(f"Features: {X.shape[1]}")
print(f"Samples: {X.shape[0]}")
print(f"Classes: {len(np.unique(y))}")

### 2.1 การ Normalize ข้อมูล

In [ ]:
# StandardScaler (Z-score normalization)
scaler_std = StandardScaler()
X_train_std = scaler_std.fit_transform(X_train)
X_test_std = scaler_std.transform(X_test)

# MinMaxScaler (0-1 normalization)
scaler_minmax = MinMaxScaler()
X_train_minmax = scaler_minmax.fit_transform(X_train)
X_test_minmax = scaler_minmax.transform(X_test)

print("=== Original Data (Feature 0) ===")
print(f"Mean: {X_train[:, 0].mean():.4f}, Std: {X_train[:, 0].std():.4f}")
print(f"Min: {X_train[:, 0].min():.4f}, Max: {X_train[:, 0].max():.4f}")

print("\n=== StandardScaler (Feature 0) ===")
print(f"Mean: {X_train_std[:, 0].mean():.4f}, Std: {X_train_std[:, 0].std():.4f}")

print("\n=== MinMaxScaler (Feature 0) ===")
print(f"Min: {X_train_minmax[:, 0].min():.4f}, Max: {X_train_minmax[:, 0].max():.4f}")

## 3. การเลือกสถาปัตยกรรม

In [ ]:
def count_parameters(layer_sizes):
    """คำนวณจำนวน parameters"""
    total = 0
    for i in range(len(layer_sizes) - 1):
        weights = layer_sizes[i] * layer_sizes[i+1]
        biases = layer_sizes[i+1]
        total += weights + biases
    return total

# เปรียบเทียบสถาปัตยกรรมต่างๆ
architectures = [
    [4, 8, 3],           # Small
    [4, 16, 8, 3],       # Medium
    [4, 32, 16, 8, 3],   # Large
]

print("=== Architecture Comparison ===")
for arch in architectures:
    params = count_parameters(arch)
    print(f"{arch}: {params} parameters")

## 4. Learning Rate Schedules

In [ ]:
def constant_lr(epoch, initial_lr):
    """Constant learning rate"""
    return initial_lr

def step_decay(epoch, initial_lr, drop_rate=0.5, epochs_drop=10):
    """Step decay"""
    return initial_lr * (drop_rate ** (epoch // epochs_drop))

def exponential_decay(epoch, initial_lr, decay_rate=0.95):
    """Exponential decay"""
    return initial_lr * (decay_rate ** epoch)

def cosine_annealing(epoch, initial_lr, total_epochs):
    """Cosine annealing"""
    return initial_lr * 0.5 * (1 + np.cos(np.pi * epoch / total_epochs))

# Visualize
epochs = 100
initial_lr = 0.1

x = range(epochs)
y_constant = [constant_lr(e, initial_lr) for e in x]
y_step = [step_decay(e, initial_lr) for e in x]
y_exp = [exponential_decay(e, initial_lr) for e in x]
y_cosine = [cosine_annealing(e, initial_lr, epochs) for e in x]

plt.figure(figsize=(10, 6))
plt.plot(x, y_constant, label='Constant')
plt.plot(x, y_step, label='Step Decay')
plt.plot(x, y_exp, label='Exponential Decay')
plt.plot(x, y_cosine, label='Cosine Annealing')
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.title('Learning Rate Schedules')
plt.legend()
plt.show()

## 5. Learning Curves

In [ ]:
def plot_learning_curves(train_losses, val_losses, title):
    """Plot learning curves"""
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Training Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(title)
    plt.legend()
    plt.show()

# Simulate learning curves
epochs = 100
train_loss_good = 1 / (1 + np.exp(-0.05 * (np.arange(epochs) - 20))) * 0.3 + 0.1
val_loss_good = train_loss_good + 0.05

train_loss_overfit = 1 / (1 + np.exp(-0.1 * (np.arange(epochs) - 10))) * 0.3 + 0.05
val_loss_overfit = np.concatenate([train_loss_overfit[:50] + 0.05, 
                                   train_loss_overfit[50:] + 0.1 + np.linspace(0, 0.3, 50)])

print("=== Good Fit ===")
plot_learning_curves(train_loss_good, val_loss_good, 'Good Fit')

print("=== Overfitting ===")
plot_learning_curves(train_loss_overfit, val_loss_overfit, 'Overfitting')

## 6. Hyperparameter Tuning

In [ ]:
# Grid Search Example
def grid_search_demo():
    """Demonstrate grid search concept"""
    learning_rates = [0.001, 0.01, 0.1, 0.5]
    hidden_sizes = [8, 16, 32, 64]
    
    print("=== Grid Search Space ===")
    print(f"Learning rates: {learning_rates}")
    print(f"Hidden sizes: {hidden_sizes}")
    print(f"Total combinations: {len(learning_rates) * len(hidden_sizes)}")
    
    # Simulate results
    print("\nSimulated Results:")
    best_acc = 0
    best_params = None
    
    for lr in learning_rates:
        for hs in hidden_sizes:
            # Simulate accuracy
            acc = 0.7 + 0.2 * np.exp(-((lr - 0.1)**2) / 0.01) * (1 - np.exp(-hs/20))
            acc += np.random.uniform(-0.02, 0.02)
            print(f"lr={lr}, hidden={hs}: accuracy={acc:.4f}")
            
            if acc > best_acc:
                best_acc = acc
                best_params = (lr, hs)
                
    print(f"\nBest: lr={best_params[0]}, hidden={best_params[1]}, accuracy={best_acc:.4f}")

grid_search_demo()

## 7. แบบฝึกหัดการคำนวณ

### แบบฝึกหัดที่ 1: คำนวณ Parameters

In [ ]:
# จงคำนวณจำนวน parameters ของ MLP ที่มีโครงสร้าง [784, 256, 128, 10]

layer_sizes = [784, 256, 128, 10]
total = count_parameters(layer_sizes)

print(f"Architecture: {layer_sizes}")
print(f"Total parameters: {total:,}")

### แบบฝึกหัดที่ 2: Learning Rate Decay

In [ ]:
# ให้ initial_lr = 0.1, decay_rate = 0.9
# จงคำนวณ learning rate ที่ epoch 0, 5, 10, 20

initial_lr = 0.1
decay_rate = 0.9
epochs_to_check = [0, 5, 10, 20]

for epoch in epochs_to_check:
    lr = initial_lr * (decay_rate ** epoch)
    print(f"Epoch {epoch}: lr = {lr:.6f}")

### แบบฝึกหัดที่ 3: Batch Size Effect

In [ ]:
# ให้ dataset มี 1000 samples
# จงคำนวณจำนวน iterations ต่อ epoch สำหรับ batch_size = 16, 32, 64, 128

n_samples = 1000
batch_sizes = [16, 32, 64, 128]

print(f"Dataset size: {n_samples}")
print("\nIterations per epoch:")
for bs in batch_sizes:
    iterations = n_samples // bs
    print(f"Batch size {bs}: {iterations} iterations")

### แบบฝึกหัดที่ 4: Normalize Data

In [ ]:
# ให้ข้อมูล x = [10, 20, 30, 40, 50]
# จง normalize ด้วย Z-score และ MinMax

x = np.array([10, 20, 30, 40, 50])

# Z-score normalization
x_zscore = (x - x.mean()) / x.std()

# MinMax normalization
x_minmax = (x - x.min()) / (x.max() - x.min())

print(f"Original: {x}")
print(f"Z-score: {x_zscore}")
print(f"MinMax: {x_minmax}")

## บทสรุป

Notebook นี้ครอบคลุม:
1. **Data Preparation**: Normalization, Train/Val/Test split
2. **Architecture Selection**: Parameter counting
3. **Learning Rate Schedules**: Constant, Step, Exponential, Cosine
4. **Learning Curves**: Diagnosing bias/variance
5. **Hyperparameter Tuning**: Grid Search